<a href="https://colab.research.google.com/github/anindyaganguly-source/Drosophila_GRN_analysis/blob/main/example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ATTENTION: only run this cell when on google colab
!git clone https://github.com/philshiu/Drosophila_brain_model.git
!pip install brian2
%cd Drosophila_brain_model

Cloning into 'Drosophila_brain_model'...
remote: Enumerating objects: 183, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 183 (delta 47), reused 44 (delta 44), pack-reused 134 (from 1)
Receiving objects: 100% (183/183), 185.06 MiB | 22.50 MiB/s, done.
Resolving deltas: 100% (83/83), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 22.4 MB/s eta 0:00:00
/content/Drosophila_brain_model


In [4]:
from model import run_exp
from model import default_params as params
import utils as utl
from brian2 import Hz

config = {
    'path_res'  : './results/example',                              # directory to store results
    'path_comp' : './2023_03_23_completeness_630_final.csv',        # csv of the complete list of Flywire neurons
    'path_con'  : './2023_03_23_connectivity_630_final.parquet',    # connectivity data
    'n_proc'    : -1,                                               # number of CPU cores (-1: use all)
}

# Introduction
## Underlying connectivity data
The connectivity of the fly brain is stored in the folowing files:
- neurons present: `config['path_comp']`
- connectivity between neurons: `config['path_con]`

## Model parameters
The equation and constants for the leaky integrate and fire model are defined
in the dictionary `default_params` in the beginning of the file `model.py`:

```
default_params = {
    # trials
    't_run'     : 1000 * ms,              # duration of trial
    'n_run'     : 30,                     # number of runs

    'v_0'       : -52 * mV,               # resting potential
    'v_rst'     : -52 * mV,               # reset potential after spike
    [...]
```
We can also change values
and pass the modified dictionary to the model (see Experiment 1).

## Addressing neurons
Here, we want to stimulate some sugar-sensing neurons in the right hemisphere.
The neurons of interest are defined via their flywire IDs:

In [5]:
neu_sugar = [
    720575940624963786,
    720575940630233916,
    720575940637568838,
    720575940638202345,
    720575940617000768,
    720575940630797113,
    720575940632889389,
    720575940621754367,
    720575940621502051,
    720575940640649691,
    720575940639332736,
    720575940616885538,
    720575940639198653,
    720575940620900446,
    720575940617937543,
    720575940632425919,
    720575940633143833,
    720575940612670570,
    720575940628853239,
    720575940629176663,
    720575940611875570,
]

For an easier identification, we define also a mapping from the flywire IDs to custom
names. The above neurons are calles `sugar_1`, `sugar_2` etc:

In [6]:
flyid2name = { f: f'sugar_{i+1}' for i, f in enumerate(neu_sugar) }
flyid2name

{720575940624963786: 'sugar_1',
 720575940630233916: 'sugar_2',
 720575940637568838: 'sugar_3',
 720575940638202345: 'sugar_4',
 720575940617000768: 'sugar_5',
 720575940630797113: 'sugar_6',
 720575940632889389: 'sugar_7',
 720575940621754367: 'sugar_8',
 720575940621502051: 'sugar_9',
 720575940640649691: 'sugar_10',
 720575940639332736: 'sugar_11',
 720575940616885538: 'sugar_12',
 720575940639198653: 'sugar_13',
 720575940620900446: 'sugar_14',
 720575940617937543: 'sugar_15',
 720575940632425919: 'sugar_16',
 720575940633143833: 'sugar_17',
 720575940612670570: 'sugar_18',
 720575940628853239: 'sugar_19',
 720575940629176663: 'sugar_20',
 720575940611875570: 'sugar_21'}

# Running simulations
## Activating a set of neurons
To run a simulation exciting these nerons we have to call `run_exp` supplying the following:
- unique name for the simulation: `exp_name`
- a list of neurons we want to stimulate: `neu_sugar`
- the connectivity data: `config['path_comp']` and `config['path_con]`
- path to store the output: `config['path_res']`
- number of CPU cores use: `config['n_procs]`

Note that running this on Google Colab can take roughly 20 minutes; it is substantially faster on a local install, depending on the number of CPU cores. By default, the neurons are excited at 200 Hz.

In [7]:
# activate sugar sensing neurons
run_exp(exp_name='sugarR', neu_exc=neu_sugar, **config)

>>> Skipping experiment sugarR because results/example/sugarR.parquet exists and force_overwrite = False


The `.parquet` file created during a simulation contains all spikes events of all neurons in the model.
We load the data again from disk by passing a list of result files to the `utl.load_exps` function.

We can see from the size of the dataframe
that more than 400 000 spikes were generated by activating the sugar neurons (30 trials, 1 s each).

In [8]:
# load data from disk
df_spike = utl.load_exps([ './results/example/sugarR.parquet' ])
df_spike

,t,trial,flywire_id,exp_name
0,0.2562,0,720575940605513649,sugarR
1,0.3607,0,720575940605513649,sugarR
2,0.6044,0,720575940605513649,sugarR
3,0.7128,0,720575940605513649,sugarR
4,0.8627,0,720575940605513649,sugarR
...,...,...,...,...
511561,0.9133,29,720575940660229505,sugarR
511562,0.9245,29,720575940660229505,sugarR
511563,0.9437,29,720575940660229505,sugarR
511564,0.9687,29,720575940660229505,sugarR


The spike times can be converted to spike rates [Hz] via `utl.get_rate`, which requires the duration of each trial.
`utl.get_rate` returns `pandas.DataFrame` objects:
1. spike rate for each neuron (rows) in each experiment (column): `df_rate`
2. standard deviation of rate across trials: `df_rate_std`

For convenience, we can optionally pass the `flyid2name` dictionary to `utl.get_rate` in order to convert flywire IDs into
meaningful names.

We can see that only about 400 neurons show activity during the simulations.

In [9]:
# calculate spike rate and standard deviation
df_rate, df_rate_std = utl.get_rate(df_spike, t_run=params['t_run'], n_run=params['n_run'], flyid2name=flyid2name)
# sort by spike rate
df_rate.sort_values('sugarR', ascending=False)

exp_name,name,sugarR
flyid,,
720575940637568838,sugar_3,202.066667
720575940621502051,sugar_9,200.200000
720575940639198653,sugar_13,199.966667
720575940617937543,sugar_15,199.400000
720575940633143833,sugar_17,199.033333
...,...,...
720575940623614442,,0.033333
720575940625852464,,0.033333
720575940614465478,,0.033333


In [10]:
# Find neurons whose assigned name contains "MN9"
df_rate[
    df_rate["name"].astype(str).str.contains("MN9", case=False, na=False)
].sort_values("sugarR", ascending=False)

exp_name,name,sugarR
flyid,,


In [11]:
# Inspect the neuron-name mapping
print(type(flyid2name))
print("Number of entries:", len(flyid2name))

# Show first 30 entries
list(flyid2name.items())[:30]

<class 'dict'>
Number of entries: 21


[(720575940624963786, 'sugar_1'),
 (720575940630233916, 'sugar_2'),
 (720575940637568838, 'sugar_3'),
 (720575940638202345, 'sugar_4'),
 (720575940617000768, 'sugar_5'),
 (720575940630797113, 'sugar_6'),
 (720575940632889389, 'sugar_7'),
 (720575940621754367, 'sugar_8'),
 (720575940621502051, 'sugar_9'),
 (720575940640649691, 'sugar_10'),
 (720575940639332736, 'sugar_11'),
 (720575940616885538, 'sugar_12'),
 (720575940639198653, 'sugar_13'),
 (720575940620900446, 'sugar_14'),
 (720575940617937543, 'sugar_15'),
 (720575940632425919, 'sugar_16'),
 (720575940633143833, 'sugar_17'),
 (720575940612670570, 'sugar_18'),
 (720575940628853239, 'sugar_19'),
 (720575940629176663, 'sugar_20'),
 (720575940611875570, 'sugar_21')]

In [12]:
# FlyWire v630 ID for left MN9
MN9 = 720575940660219265

# Get MN9 firing rate following sugar-GRN stimulation
df_rate.loc[[MN9]]

exp_name,name,sugarR
flyid,,
720575940660219265,,93.266667


In [13]:
# ==========================================
# WHOLE-BRAIN RESPONSE TO SUGAR STIMULATION
# ==========================================

# Remove the directly stimulated sugar GRNs
downstream = df_rate.drop(
    index=[x for x in neu_sugar if x in df_rate.index]
).copy()

# Rename the firing-rate column for clarity
downstream = downstream.rename(
    columns={'sugarR': 'mean_rate_Hz'}
)

# Add trial-to-trial SD
downstream['SD_Hz'] = df_rate_std.loc[
    downstream.index, 'sugarR'
]

# Sort strongest responses first
downstream = downstream.sort_values(
    'mean_rate_Hz',
    ascending=False
)

# Add rank
downstream.insert(
    0,
    'rank',
    range(1, len(downstream) + 1)
)

print("Number of downstream neurons that fired:",
      len(downstream))

print("\nTop 50 downstream neurons:")
downstream.head(50)

Number of downstream neurons that fired: 427

Top 50 downstream neurons:


exp_name,rank,name,mean_rate_Hz,SD_Hz
flyid,,,,
720575940622695448,1,,156.766667,1.838175
720575940629888530,2,,146.500000,2.217356
720575940627383685,3,,141.233333,1.667000
720575940618165019,4,,128.566667,9.354797
720575940619973712,5,,125.866667,1.257864
720575940615041430,6,,124.800000,2.725191
720575940630868793,7,,124.033333,9.199577
720575940629778554,8,,116.200000,7.304793
720575940626191306,9,,115.800000,1.620699


In [14]:
# Save the complete sugar downstream response
sugar_downstream = downstream.copy()

# Flag MN9
sugar_downstream["MN9"] = sugar_downstream.index == MN9

# Where is MN9?
sugar_downstream.loc[[MN9]]

exp_name,rank,name,mean_rate_Hz,SD_Hz,MN9
flyid,,,,,
720575940660219265,18,,93.266667,3.151014,True


In [15]:
# Download FlyWire annotation table corresponding to the v630-era release
!wget -q https://github.com/flyconnectome/flywire_annotations/raw/1.1.0/supplemental_files/Supplemental_file1_neuron_annotations.tsv -O flywire_annotations_v630.tsv

print("Downloaded v630 annotation table")

Downloaded v630 annotation table


In [17]:
import os

fname = "flywire_annotations_v630.tsv"

print("File exists:", os.path.exists(fname))
print("File size:", os.path.getsize(fname) if os.path.exists(fname) else "NA")

File exists: True
File size: 0


In [18]:
!rm -f flywire_annotations_v630.tsv

In [21]:
# Clone the correct historical annotation release for FlyWire v630
!rm -rf flywire_ann_v630

!git clone --branch v1.1.0 --depth 1 \
    https://github.com/flyconnectome/flywire_annotations.git \
    flywire_ann_v630

# Check the files
!find flywire_ann_v630/supplemental_files -maxdepth 1 -type f | sort

Cloning into 'flywire_ann_v630'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 23 (delta 0), reused 15 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (23/23), 10.06 MiB | 8.28 MiB/s, done.
Note: switching to 'df6bb136f5b3d91c3992df4e8de2642329e2a384'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

flywire_ann_v630/supplemental_files/Supplemental_file1_annotations.tsv
flywire_ann_v630/supplemental_files/Supplemen

In [23]:
import os

print("Repository exists:",
      os.path.exists("flywire_ann_v630"))

for root, dirs, files in os.walk("flywire_ann_v630"):
    for f in files:
        if f.endswith((".tsv", ".csv", ".feather")):
            print(os.path.join(root, f))

Repository exists: True
flywire_ann_v630/supplemental_files/Supplemental_file3_hemilineages_clustering.csv
flywire_ann_v630/supplemental_files/Supplemental_file1_annotations.tsv
flywire_ann_v630/supplemental_files/Supplemental_file4_hemibrain_meta.csv
flywire_ann_v630/supplemental_files/Supplemental_file2_summary_with_ngl_links.csv


In [24]:
import pandas as pd

ann = pd.read_csv(
    "flywire_ann_v630/supplemental_files/Supplemental_file1_annotations.tsv",
    sep="\t",
    low_memory=False
)

print("Rows:", len(ann))
print("\nColumns:")
print(ann.columns.tolist())

ann.head()

Rows: 128824

Columns:
['supervoxel_id', 'root_id', 'pos_x', 'pos_y', 'pos_z', 'soma_x', 'soma_y', 'soma_z', 'nucleus_id', 'flow', 'super_class', 'cell_class', 'cell_sub_class', 'cell_type', 'hemibrain_type', 'ito_lee_hemilineage', 'hartenstein_hemilineage', 'morphology_group', 'top_nt', 'top_nt_conf', 'side', 'nerve', 'fbbt_id', 'status']


,supervoxel_id,root_id,pos_x,pos_y,pos_z,soma_x,soma_y,soma_z,nucleus_id,flow,...,hemibrain_type,ito_lee_hemilineage,hartenstein_hemilineage,morphology_group,top_nt,top_nt_conf,side,nerve,fbbt_id,status
0,78112261444987077,720575940628857210,109306,50491,3960,104904.0,47464.0,5461.0,2453924.0,intrinsic,...,PS180,SMPpv2_ventral,CP1_ventral,SMPpv2_ventral_3,acetylcholine,0.914499,left,NaN,FBbt_20001935,NaN
1,82475466912542440,720575940626838909,172029,55635,1592,177472.0,56936.0,1429.0,7393349.0,intrinsic,...,NaN,VLPl2_medial,BLAv2_medial,VLPl2_medial_1,acetylcholine,0.638088,right,NaN,NaN,NaN
2,83038623024880664,720575940626046919,180632,58664,1925,180632.0,58664.0,1925.0,7415038.0,intrinsic,...,AVLP429,NaN,NaN,NaN,acetylcholine,0.838454,right,NaN,FBbt_20000538,NaN
3,79801523353604463,720575940630311383,133800,56063,1847,180728.0,61008.0,1630.0,7415013.0,intrinsic,...,AVLP151,putative_primary,putative_primary,NaN,acetylcholine,0.755116,right,NaN,FBbt_20000260,NaN
4,83038554439606237,720575940633370649,180496,57448,2989,180496.0,57448.0,2989.0,7415848.0,intrinsic,...,LC27,NaN,NaN,NaN,acetylcholine,0.886547,right,NaN,FBbt_00051248,NaN


## Change stimulation frequency

We want to change the frequency of the stimulation of the sugar neurons.
To do so we modify the value for `r_poi` in the `default_params` dictionary and pass the altered dictionary to the `run_exp` function.

Note: Since physical quantities in `brian2` have to have the correct unit, we also need the `brian2.Hz` object
to define a frequency.

In [ ]:
# run with different frequency
params['r_poi'] = 100 * Hz

run_exp(exp_name='sugarR_100Hz', neu_exc=neu_sugar, params=params, **config)

We load the results via the `utl.load_exps` function and convert the spike events to rates with `utl.get_rate`

In [ ]:
ps = [
    './results/example/sugarR.parquet',
    './results/example/sugarR_100Hz.parquet',
]

df_spike = utl.load_exps(ps)
df_rate, df_rate_std = utl.get_rate(df_spike, t_run=params['t_run'], n_run=params['n_run'], flyid2name=flyid2name)
df_rate.sort_values('sugarR_100Hz', ascending=False, inplace=True)
df_rate

## Silencing neurons
We want to silence the most active neurons individually to see how that changes the activity patterns.
We do so by passing the neuron IDs we want to silence as a list `run_exp` via the `neu_slnc` argument.
In the following example, we are silencing a single neuron `[ i ]` while exciting the sugar neurons `neu_sugar`.
We can then investigate how silencing of each individual neuron affects the firing rate of a given neuron, say, MN9.

In [ ]:
#First, let's check on the MN9 firing rate when no neurons are silenced.
id_mn9 = 720575940660219265 #id for MN9
x = df_rate.loc[id_mn9, "sugarR_100Hz"]
print(f'Rate for neuron {id_mn9} is {x}')

In [ ]:
# IDs of 3 most active neurons. These neurons are all sugar-sensing neurons.
ids = df_rate.sort_values('sugarR_100Hz', ascending=False).index[:3]

for i in ids:
    run_exp(exp_name=f'sugarR-{i}', neu_exc=neu_sugar, neu_slnc=[ i ], params=params, **config)

In [ ]:
# output files
ps = [ f'./results/example/sugarR-{i}.parquet' for i in ids ]

# calculate spike rate and sort
df_spike = utl.load_exps(ps)
df_rate, df_rate_std = utl.get_rate(df_spike, t_run=params['t_run'], n_run=params['n_run'])
df_rate.loc[id_mn9, :].sort_values(ascending=True)